# Coastal Dynamics — Tarragona Coast, Spain
## A Cloud-Native Time Series Analysis of Shoreline Change

**Open Science in Practice — FERS School, Bertinoro, April 2026**

---

### Scientific Questions

>Has the coastline near Tarragona (NE Spain) changed in extent between 2017 and 2024?

>How have sea level anomaly and significant wave height behaved over the same period?

>Does the choice of water index (NDWI vs MNDWI) affect the detected change signal?

### Study Site

The study area (`bbox [0.55, 40.55, 1.05, 40.90]`) covers the Tarragona coastal zone
in Catalonia, NE Spain — including the Costa Daurada beaches, the Ebro Delta margin,
and nearshore Mediterranean waters. The area is sensitive to storm surges, longshore
sediment transport, and seasonal wave climatology.

### Methods: two water indices compared

| Index | Formula | Bands | Reference |
|---|---|---|---|
| **NDWI** | (Green − NIR) / (Green + NIR) | B03, B08 | McFeeters (1996) |
| **MNDWI** | (Green − SWIR) / (Green + SWIR) | B03, B11 | Xu (2006) |

Both indices range from −1 to +1. The water/land threshold is **fixed at 0** for both:
- index ≥ 0 → water
- index < 0 → land / beach / built-up

A fixed threshold of 0 is theoretically motivated (the original definition of both indices)
and empirically justified by inspecting the NDWI/MNDWI histograms of all three scenes
(Section 3b), which show the water–land transition consistently crossing zero.
An adaptive Otsu threshold was tested in v2 but produced a misplaced threshold for the
January 2021 scene due to different illumination geometry, leading to overestimation
of land area. The fixed threshold is more robust across scenes from different months.

### Data sources (all open, cloud-native)
| Variable | Source | Access |
|---|---|---|
| Multispectral imagery | Sentinel-2 L2A | Planetary Computer STAC |
| Sea Level Anomaly (SLA) | CMEMS SEALEVEL_GLO | `copernicusmarine` Python client |
| Significant Wave Height | CMEMS Wave reanalysis | `copernicusmarine` Python client |
| Tidal level at acquisition | REDMAR Tarragona (3756N) | Puertos del Estado annual reports |



---
## Section 0 — Environment Setup

In [ ]:
# Core scientific stack
import numpy as np
import xarray as xr
import rioxarray
import pandas as pd
import geopandas as gpd
import warnings

# STAC access
import pystac_client
import planetary_computer

# CMEMS access
import copernicusmarine

# Raster utilities
from rioxarray.merge import merge_arrays
from rasterio.features import shapes
from shapely.geometry import shape
from scipy.ndimage import binary_opening

# Visualization
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import panel as pn
import matplotlib.pyplot as plt
import folium

hv.extension('bokeh')
pn.extension()

print('✅ All imports successful')

---
## Section 1 — Study Area Definition

In [ ]:
# Study area — Tarragona coast, NE Spain
BBOX            = [0.55, 40.55, 1.05, 40.90]   # [lon_min, lat_min, lon_max, lat_max]
YEARS           = list(range(2017, 2026))        # 2017–2025
CLOUD_COVER_MAX = 20                             # %
ANALYSIS_YEARS  = [2017, 2021, 2024]             # years used for index computation
FIXED_THRESHOLD = 0.0                            # water/land threshold for both NDWI and MNDWI

# Seasonal window: Jan–Apr — consistent illumination, low wave energy
DATE_WINDOWS = [(f'{y}-01-01', f'{y}-04-30') for y in YEARS]

print(f'Study area bbox:   {BBOX}')
print(f'Search window:     Jan–Apr each year')
print(f'Analysis years:    {ANALYSIS_YEARS}')
print(f'Fixed threshold:   {FIXED_THRESHOLD}  (index < 0 → land, index ≥ 0 → water)')

In [ ]:
# Quick map to verify study area
center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
folium.Rectangle(
    bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
    color='red', fill=True, fill_opacity=0.2
).add_to(m)
m

---
## Section 2 — Sentinel-2 Scene Search

In [ ]:
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)
print('✅ Connected to Planetary Computer STAC catalog')

In [ ]:
# Discover which MGRS tiles cover our bbox
discover = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime='2023-01-01/2023-12-31',
    query={'eo:cloud_cover': {'lt': CLOUD_COVER_MAX}}
)
TILES = sorted({item.properties['s2:mgrs_tile'] for item in discover.items()})
print(f'Tiles covering the study area: {TILES}')

In [ ]:
# Search best scene (least cloudy) per tile per year within Jan–Apr
best_items = {}   # {year: {tile: item}}

for year, (date_start, date_end) in zip(YEARS, DATE_WINDOWS):
    best_items[year] = {}
    for tile in TILES:
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=BBOX,
            datetime=f'{date_start}/{date_end}',
            query={
                'eo:cloud_cover': {'lt': CLOUD_COVER_MAX},
                's2:mgrs_tile':   {'eq': tile}
            }
        )
        items = list(search.items())
        if items:
            best = min(items, key=lambda x: x.properties['eo:cloud_cover'])
            best_items[year][tile] = best
            print(f"{year}  tile {tile}: {best.datetime.date()} — cloud {best.properties['eo:cloud_cover']:.4f}%")
        else:
            print(f"{year}  tile {tile}: ⚠️  no scenes found")

In [ ]:
# Verify 2017 scene is Jan–Apr; widen cloud threshold if needed
print('=== 2017 scene check ===')
for tile, item in best_items[2017].items():
    month = item.datetime.month
    flag  = '✅' if month in [1, 2, 3, 4] else '⚠️  SEASONAL MISMATCH'
    print(f'  tile {tile}: {item.datetime.date()}  month={month}  {flag}')

for tile in TILES:
    if tile not in best_items[2017] or best_items[2017][tile].datetime.month > 4:
        print(f'\n⚠️  Widening cloud threshold for 2017 tile {tile}...')
        search_wide = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=BBOX,
            datetime='2017-01-01/2017-04-30',
            query={'eo:cloud_cover': {'lt': 50}, 's2:mgrs_tile': {'eq': tile}}
        )
        items_wide = list(search_wide.items())
        if items_wide:
            best_wide = min(items_wide, key=lambda x: x.properties['eo:cloud_cover'])
            best_items[2017][tile] = best_wide
            print(f'  → Replaced with: {best_wide.datetime.date()}  cloud {best_wide.properties["eo:cloud_cover"]:.2f}%')
        else:
            print(f'  → No scenes found even at 50% cloud threshold')

---
## Section 3 — Load Bands and Compute Indices

### 3a — Band loading and index computation

In [ ]:
def load_band_mosaic(year, band, overview_level=2):
    """Load a band across all tiles for a year, mosaic, and clip to BBOX.
    Items are re-signed immediately before reading to avoid expired SAS tokens.
    """
    tiles_data = []
    for tile, item in best_items[year].items():
        signed = planetary_computer.sign(item)   # fresh token every call
        da = rioxarray.open_rasterio(
            signed.assets[band].href,
            overview_level=overview_level
        ).squeeze()
        tiles_data.append(da)
    mosaic = merge_arrays(tiles_data) if len(tiles_data) > 1 else tiles_data[0]
    return mosaic.rio.clip_box(*BBOX, crs='EPSG:4326')

In [ ]:
ndwi_annual  = {}
mndwi_annual = {}
rgb_annual   = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, year in zip(axes, ANALYSIS_YEARS):

    # Load bands
    red   = load_band_mosaic(year, 'B04', overview_level=2)   # 10m Red
    green = load_band_mosaic(year, 'B03', overview_level=2)   # 10m Green
    blue  = load_band_mosaic(year, 'B02', overview_level=2)   # 10m Blue
    nir   = load_band_mosaic(year, 'B08', overview_level=2)   # 10m NIR
    swir  = load_band_mosaic(year, 'B11', overview_level=1)   # 20m SWIR native

    # Reproject SWIR (20m) to match green (10m) grid
    swir_10m = swir.rio.reproject_match(green)

    g = green.astype(float)
    n = nir.astype(float)
    s = swir_10m.astype(float)

    source_date = str(next(iter(best_items[year].values())).datetime.date())

    # NDWI: (Green − NIR) / (Green + NIR)   McFeeters 1996
    ndwi = (g - n) / (g + n)
    ndwi.name = 'NDWI'
    ndwi.attrs.update({'long_name': f'NDWI {year}', 'source_date': source_date})

    # MNDWI: (Green − SWIR) / (Green + SWIR)   Xu 2006
    mndwi = (g - s) / (g + s)
    mndwi.name = 'MNDWI'
    mndwi.attrs.update({'long_name': f'MNDWI {year}', 'source_date': source_date})

    ndwi_annual[year]  = ndwi
    mndwi_annual[year] = mndwi
    rgb_annual[year]   = xr.concat([red, green, nir], dim='band')

    # RGB preview
    rgb = np.stack([red.values, green.values, blue.values], axis=-1).astype(float)
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    ax.imshow(rgb)
    ax.set_title(f'{year}  |  {source_date}', fontsize=10, fontweight='bold')
    ax.axis('off')

    print(f'✅ {year} ({source_date})')
    print(f'   NDWI  [{float(ndwi.min()):.3f}, {float(ndwi.max()):.3f}]')
    print(f'   MNDWI [{float(mndwi.min()):.3f}, {float(mndwi.max()):.3f}]')

plt.suptitle('Sentinel-2 RGB — Tarragona Coast — 2017 / 2021 / 2024', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3b — Threshold justification: histogram analysis

Each histogram shows the distribution of index values across all pixels in the scene.
A well-behaved water/land index should be **bimodal**: one peak for water (high values)
and one for land (low values), with the threshold placed in the valley between them.

The red dashed line shows the **fixed threshold = 0** used in this analysis.
The orange dotted line shows where the **Otsu adaptive threshold** would be placed.

For 2017 and 2024 the two thresholds are reasonably close. For 2021 (January acquisition),
Otsu places the threshold substantially above zero (NDWI = 0.317, MNDWI = 0.294) due to
a shifted distribution caused by lower sun angle and different water reflectance in winter.
The fixed threshold at 0 is more consistent across all three scenes.

In [ ]:
from skimage.filters import threshold_otsu

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for col, year in enumerate(ANALYSIS_YEARS):
    date = ndwi_annual[year].attrs['source_date']

    for row, (label, index_data) in enumerate([
        ('NDWI',  ndwi_annual[year]),
        ('MNDWI', mndwi_annual[year])
    ]):
        ax   = axes[row, col]
        vals = index_data.values.flatten()
        vals = vals[np.isfinite(vals)]

        t_otsu = threshold_otsu(vals)

        ax.hist(vals, bins=250, color='steelblue', alpha=0.75, density=True)
        ax.axvline(FIXED_THRESHOLD, color='red',   linestyle='--', linewidth=2,
                   label=f'Fixed = {FIXED_THRESHOLD}')
        ax.axvline(t_otsu,          color='orange', linestyle=':',  linewidth=2,
                   label=f'Otsu  = {t_otsu:.3f}')

        ax.set_xlim(-1, 1)
        ax.set_xlabel(f'{label} value', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'{label} — {year}\n{date}', fontsize=9, fontweight='bold')
        ax.legend(fontsize=8)

        # Annotate problematic Otsu placement for 2021
        if year == 2021 and abs(t_otsu - FIXED_THRESHOLD) > 0.05:
            ax.annotate(
                'Otsu shifted\n(winter scene)',
                xy=(t_otsu, ax.get_ylim()[1] * 0.6),
                xytext=(t_otsu + 0.15, ax.get_ylim()[1] * 0.75),
                arrowprops=dict(arrowstyle='->', color='orange'),
                fontsize=8, color='orange'
            )

plt.suptitle(
    'Index histograms — Fixed threshold (red) vs Otsu adaptive threshold (orange)\n'
    'Fixed threshold = 0 is more consistent across all scenes',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# Visual comparison of NDWI vs MNDWI maps
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, year in enumerate(ANALYSIS_YEARS):
    date = ndwi_annual[year].attrs['source_date']

    im0 = axes[0, col].imshow(ndwi_annual[year].values,  cmap='RdYlBu', vmin=-1, vmax=1)
    axes[0, col].set_title(f'NDWI {year}\n{date}  |  threshold = 0', fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)

    im1 = axes[1, col].imshow(mndwi_annual[year].values, cmap='RdYlBu', vmin=-1, vmax=1)
    axes[1, col].set_title(f'MNDWI {year}\n{date}  |  threshold = 0', fontsize=9)
    axes[1, col].axis('off')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

axes[0, 0].set_ylabel('NDWI', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('MNDWI', fontsize=11, fontweight='bold')

plt.suptitle('NDWI vs MNDWI — Tarragona Coast (blue=water, red=land) — fixed threshold = 0',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 4 — Land Area Quantification

Land pixels: index < 0 (fixed threshold).  
A morphological opening (2 iterations) removes isolated noisy pixels.

In [ ]:
land_area = {}   # {year: {'ndwi_ha': float, 'mndwi_ha': float}}
land_gdf  = {}   # {year: {'ndwi': GeoDataFrame, 'mndwi': GeoDataFrame}}

for year in ANALYSIS_YEARS:
    land_area[year] = {}
    land_gdf[year]  = {}

    for method, index_data in [('ndwi', ndwi_annual[year]), ('mndwi', mndwi_annual[year])]:

        # Fixed threshold = 0
        land_mask  = (index_data < FIXED_THRESHOLD).astype('uint8')

        # Morphological opening — removes isolated noisy pixels
        land_clean = binary_opening(land_mask.values, iterations=2).astype('uint8')
        land_mask_clean = xr.DataArray(
            land_clean, dims=land_mask.dims, coords=land_mask.coords
        )
        land_mask_clean.rio.write_crs(index_data.rio.crs, inplace=True)

        # Vectorise land pixels → polygons
        transform = index_data.rio.transform()
        crs       = index_data.rio.crs
        polys = []
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            for geom, val in shapes(land_clean, transform=transform):
                if val == 1:
                    polys.append(shape(geom))

        gdf = gpd.GeoDataFrame(geometry=polys, crs=crs).dissolve()
        land_gdf[year][method] = gdf

        # Area in hectares
        gdf_m   = gdf.to_crs(gdf.estimate_utm_crs())
        area_ha = gdf_m.area.sum() / 10_000
        land_area[year][f'{method}_ha'] = area_ha

    print(f'{year}: NDWI = {land_area[year]["ndwi_ha"]:.1f} ha  |  '
          f'MNDWI = {land_area[year]["mndwi_ha"]:.1f} ha')

df_area = pd.DataFrame([
    {'year': y, 'ndwi_ha': v['ndwi_ha'], 'mndwi_ha': v['mndwi_ha']}
    for y, v in land_area.items()
])
df_area

---
## Section 5 — CMEMS Oceanographic Variables

In [ ]:
# Sea Level Anomaly
ds_sla = copernicusmarine.open_dataset(
    dataset_id='cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D',
    variables=['sla'],
    minimum_longitude=BBOX[0], maximum_longitude=BBOX[2],
    minimum_latitude=BBOX[1],  maximum_latitude=BBOX[3],
    start_datetime='2017-01-01', end_datetime='2024-12-31',
)
print(ds_sla)

sla_annual_mean = (
    ds_sla['sla']
    .mean(dim=['latitude', 'longitude'])
    .resample(time='YE').mean()
)
df_sla = sla_annual_mean.to_dataframe().reset_index()
df_sla['year'] = df_sla['time'].dt.year
df_sla = df_sla[['year', 'sla']]
df_sla

In [ ]:
# Significant Wave Height
ds_waves = copernicusmarine.open_dataset(
    dataset_id='cmems_mod_glo_wav_my_0.2deg_PT3H-i',
    variables=['VHM0'],
    minimum_longitude=BBOX[0], maximum_longitude=BBOX[2],
    minimum_latitude=BBOX[1],  maximum_latitude=BBOX[3],
    start_datetime='2017-01-01', end_datetime='2024-12-31',
)
print(ds_waves)

# Storm season: Oct–Feb (peak Mediterranean wave energy)
hs = ds_waves['VHM0']
hs_storm  = hs.sel(time=hs.time.dt.month.isin([10, 11, 12, 1, 2]))
hs_annual = (
    hs_storm
    .mean(dim=['latitude', 'longitude'])
    .resample(time='YE').mean()
)
df_hs = hs_annual.to_dataframe().reset_index()
df_hs['year'] = df_hs['time'].dt.year
df_hs = df_hs[['year', 'VHM0']].rename(columns={'VHM0': 'hs_m'})
df_hs

In [ ]:
# ── Tidal level at acquisition time — REDMAR Tarragona (station 3756N) ───────
# Source: Puertos del Estado annual reports 3756N17, 3756N21, 3756N24
# Values are mean sea level (cm) at approximately 10:30 UTC on acquisition date
# Reference: gauge zero, Port of Tarragona
# URL pattern: https://bancodatos.puertos.es/BD/informes/anuales/3/3756N{YY}.pdf

tidal_df = pd.DataFrame({
    'year':     [2017,        2021,        2024],
    'date':     ['03/04/2017','17/01/2021','12/03/2024'],
    'tide_cm':  [28.5,        27.5,        30.0],
})
tidal_df['tide_anomaly_cm'] = tidal_df['tide_cm'] - tidal_df['tide_cm'].mean()

print('Tidal level at acquisition time (REDMAR Tarragona):')
print(tidal_df.to_string(index=False))
print(f"\nRange across dates: {tidal_df['tide_cm'].max() - tidal_df['tide_cm'].min():.1f} cm")
print('→ Tidal differences are negligible (<3 cm). Tidal confounding is ruled out.')

In [ ]:
# Merge all variables (analysis years only)
df = (
    df_area
    .merge(df_sla,   on='year')
    .merge(df_hs,    on='year')
    .merge(tidal_df[['year', 'tide_cm', 'tide_anomaly_cm']], on='year')
)
df['year'] = df['year'].astype(int)
print('Integrated dataset:')
df

---
## Section 6 — Interactive Dashboard

Controls:
- **Year slider** — toggles between 2017, 2021, 2024
- **Index toggle** — switches the map between NDWI and MNDWI

The map shows the index raster with the land polygon (threshold = 0) overlaid.
The right panel shows land area for both indices, plus SLA and wave height time series.

In [ ]:
year_slider   = pn.widgets.DiscreteSlider(
    name='Year', options=ANALYSIS_YEARS, value=ANALYSIS_YEARS[0]
)
method_toggle = pn.widgets.RadioButtonGroup(
    name='Index method', options=['NDWI', 'MNDWI'], value='NDWI', button_type='primary'
)

# ── LEFT PANEL: index map + land polygons ─────────────────────────────────────
@pn.depends(year_slider.param.value, method_toggle.param.value)
def index_map(year, method):
    data = ndwi_annual[year] if method == 'NDWI' else mndwi_annual[year]
    gdf  = land_gdf[year][method.lower()].to_crs('EPSG:4326')
    date = data.attrs['source_date']

    base = data.hvplot.image(
        x='x', y='y',
        cmap='RdYlBu', clim=(-1, 1),
        title=f'{method} — Tarragona {year}  |  {date}  |  threshold = 0',
        width=540, height=480,
        colorbar=True,
        clabel=f'{method} (blue=water, red=land)',
        geo=True, tiles='OSM',
    )
    polys = gdf.hvplot.polygons(
        geo=True,
        fill_color=None,
        line_color='black',
        line_width=1.5,
    )
    return base * polys


# ── RIGHT PANEL: time series ──────────────────────────────────────────────────
@pn.depends(year_slider.param.value)
def timeseries_panel(year):

    # Melt df so hvplot can use 'by' for grouping
    df_melted = df[['year', 'ndwi_ha', 'mndwi_ha']].melt(
        id_vars='year',
        value_vars=['ndwi_ha', 'mndwi_ha'],
        var_name='method',
        value_name='land_ha'
    )
    df_melted['method'] = df_melted['method'].map(
        {'ndwi_ha': 'NDWI', 'mndwi_ha': 'MNDWI'}
    )

    area_both = (
        df_melted.hvplot.line(
            x='year', y='land_ha', by='method',
            color=['#F5A623', '#8E44AD'],
            line_width=2.5,
            width=420, height=190,
            ylabel='Land area (ha)',
            title='Land area: NDWI vs MNDWI  (threshold = 0)',
        )
        * df_melted.hvplot.scatter(
            x='year', y='land_ha', by='method',
            color=['#F5A623', '#8E44AD'], size=60,
        )
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    sla_line = (
        df_sla.hvplot.line(
            x='year', y='sla',
            color='#2E86C1', line_width=2.5, label='SLA (m)',
            width=420, height=170, ylabel='SLA (m)',
            title='Sea level anomaly',
        )
        * df_sla.hvplot.scatter(x='year', y='sla', color='#2E86C1', size=40)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    hs_line = (
        df_hs.hvplot.line(
            x='year', y='hs_m',
            color='#1ABC9C', line_width=2.5, label='Hs (m)',
            width=420, height=170, ylabel='Hs (m)',
            title='Significant wave height (storm season Oct–Feb)',
        )
        * df_hs.hvplot.scatter(x='year', y='hs_m', color='#1ABC9C', size=40)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    return pn.Column(
        pn.pane.Markdown('### Time series'),
        area_both,
        sla_line,
        hs_line,
    )


# ── ASSEMBLE DASHBOARD ────────────────────────────────────────────────────────
header = pn.pane.Markdown(
    '# Tarragona Coast — Shoreline Dynamics 2017 / 2021 / 2024\n'
    '**Data:** Sentinel-2 L2A (Planetary Computer) · CMEMS SLA · CMEMS Wave reanalysis · REDMAR Tarragona  '
    '| **Indices:** NDWI (McFeeters 1996) vs MNDWI (Xu 2006) | **Threshold:** fixed = 0',
    sizing_mode='stretch_width'
)

controls = pn.Row(
    pn.Column('### Year', year_slider),
    pn.Column('### Index method', method_toggle),
)

dashboard = pn.Column(
    header,
    controls,
    pn.Row(
        pn.panel(index_map),
        pn.panel(timeseries_panel),
    ),
)

dashboard.servable()
dashboard

---
## Section 7 — Interpretation

### Threshold decision (v2 → v3 → v4)

In v2, an adaptive Otsu threshold was computed independently for each scene and index.
For 2017 the Otsu thresholds were NDWI = 0.184 and MNDWI = 0.242, and for 2024 they
were NDWI = −0.097 and MNDWI = 0.090 — both reasonably close to the physical zero
crossing, though already showing scene-to-scene variability. For 2021 (January acquisition),
Otsu placed the threshold at NDWI = 0.317 and MNDWI = 0.294 — substantially above
zero — causing large areas of open water with slightly elevated reflectance (due to lower
winter sun angle and higher water turbidity in January) to be misclassified as land.
The histogram in Section 3b confirms that the water–land distribution crosses zero
consistently across all three years regardless of scene-specific conditions, justifying
a fixed threshold of 0 as the more physically grounded and temporally consistent choice.

---

### Tidal correction check

Tidal levels at the time of Sentinel-2 acquisition were extracted from REDMAR Tarragona
annual reports (gauge station 3756N, Port of Tarragona):

| Year | Acquisition date | Sea level at ~10:30 UTC |
|---|---|---|
| 2017 | 03 April 2017 | 28.5 cm |
| 2021 | 17 January 2021 | 27.5 cm |
| 2024 | 12 March 2024 | 30.0 cm |

The range across acquisition dates is only 2.5 cm — well within the noise of any
pixel-level classification at 10–20m resolution, and negligible on the gently-sloping
beaches and Ebro Delta flats that dominate the study area. Tidal state is therefore
ruled out as a meaningful confounding factor for the observed differences in classified
land area.

---

### Index comparison

MNDWI consistently detected less land area than NDWI across all three years, with
differences of 868 ha (2017), 988 ha (2021), and 1,083 ha (2024). This systematic offset
is expected: MNDWI uses SWIR instead of NIR, which is more strongly absorbed by water
and more sensitive to moisture in sand and soil, producing a sharper water–land contrast
and classifying marginally wet intertidal areas as water rather than land. As a result,
MNDWI provides a more conservative — and likely more accurate — estimate of dry land
extent in a coastal setting.

The largest discrepancy between the two indices occurs in 2024 (1,083 ha), likely because
the lower overall land area in that year means the boundary pixels — where the two indices
diverge most — represent a proportionally larger share of the total classified surface.

Critically, both indices show a fully consistent direction of change across the three years:
2017 and 2021 are nearly identical, and 2024 shows a clear reduction relative to both.
This consistency across two independent spectral formulations strengthens confidence in
the result.

---

### Temporal change

Land area was essentially stable between 2017 and 2021 — the MNDWI difference is only
100 ha (~0.15%), and the NDWI difference is 220 ha (~0.3%). Both are within the
uncertainty of a 10–20m pixel classification, and no meaningful shoreline change can
be inferred from this contrast. The near-identical tidal levels at acquisition (28.5 vs
27.5 cm) confirm that this small difference is not a tidal artifact. It likely reflects
natural inter-annual variability in sediment distribution on the Ebro Delta, which is a
highly dynamic system subject to episodic depositional and erosional events independent
of long-term sea level trends.

The 2024 scene tells a different story. Using MNDWI, land area decreased by **1,495 ha
(−2.2%)** relative to 2017; NDWI shows a consistent reduction of **1,280 ha (−1.9%)**.
This signal cannot be attributed to tidal or seasonal artifacts — the 2024 tidal level
was actually 1.5 cm *higher* than 2017, which would if anything expose *less* land and
thus slightly *underestimate* the true reduction, meaning the observed loss is a lower
bound. The 2024 reduction coincides with the highest annual mean sea level anomaly
in the series (SLA = +0.117 m in 2024, vs +0.059 m in 2017 and +0.076 m in 2021).
A persistently elevated regional sea level reduces the area of intertidal and supratidal
surfaces exposed above the waterline, consistent with the observed reduction in classified
land extent.

Wave height shows a modest declining trend across the three years
(Hs = 0.701 m in 2017, 0.700 m in 2021, 0.629 m in 2024, storm season October–February).
Lower wave energy in 2024 would generally reduce onshore sediment transport capacity,
potentially contributing to reduced beach width and therefore smaller classified land area.
However, this effect is secondary to the sea level signal given the magnitude of the
difference, and the two drivers likely act in concert rather than independently.

---

### Limitations

- **Temporal sampling:** Only three time points spanning seven years. This is
  insufficient for formal trend detection or statistical significance testing.
  A robust shoreline change analysis would require annual or sub-annual observations
  over at least a decade.

- **Seasonal residual bias:** Despite restricting acquisition to January–April, the
  three scenes span different months (April 2017, January 2021, March 2024). January
  scenes have lower solar elevation and potentially different atmospheric and water
  surface conditions relative to April, which may introduce a residual illumination
  bias not fully corrected by the fixed threshold. The Otsu threshold values
  (NDWI: 0.184 → 0.317 → −0.097 across years) reflect this scene-to-scene
  variability and illustrate why a fixed threshold is preferable but not a complete
  solution.

- **Spatial resolution:** At overview_level=2 (~20m effective GSD), shoreline
  displacements smaller than approximately 40m are below the detection limit. The
  Ebro Delta and Costa Daurada coastlines include dynamic narrow beach sections where
  relevant changes may be occurring at sub-pixel scale.

- **Single threshold for a heterogeneous scene:** The bbox covers a mix of coastal
  landforms — sandy beaches, Ebro Delta flats, port infrastructure, agricultural land,
  and open sea. A single threshold of 0 performs well for water/sand discrimination
  but does not distinguish between these land cover types, meaning the land area
  metric aggregates genuinely different surfaces that may respond differently to sea
  level and wave forcing.

- **No atmospheric correction validation:** Although Sentinel-2 L2A products include
  Sen2Cor atmospheric correction, residual aerosol and sun glint effects over water
  can shift index values, particularly for the winter January 2021 scene.

---

### FAIR statement

| Principle | Status | Implementation |
|---|---|---|
| **F**indable | ✅ | All datasets discoverable via STAC (Planetary Computer) and CMEMS catalog APIs with standardized metadata |
| **A**ccessible | ✅ | Sentinel-2 L2A via Planetary Computer (free, no login); CMEMS via free registered account; REDMAR data via open Puertos del Estado portal |
| **I**nteroperable | ✅ | CF-compliant spatial coordinates throughout via rioxarray; CMEMS data in standard NetCDF/Zarr with CF conventions |
| **R**eusable | ✅ | Notebook fully reproducible on any JupyterHub with Pangeo stack; all parameters (BBOX, years, threshold) defined as named constants in Section 1; no local files required |